# Preprocess — Steel Industry Energy Consumption

We do **not** drop features blindly. This notebook:
1. Loads the raw data
2. Checks quality & distributions
3. Checks correlations (target leakage + redundant features)
4. **Then** decides what to keep/drop based on what we see
5. Saves clean data for LSTM training

Run cells top to bottom.

## Setup & paths

In [ ]:

from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/Shared-Colab-Storage"
DATA_PATH = f"{BASE}/Steel_industry_data.csv"
SAVE_15MIN = f"{BASE}/agent-final/outputs/preprocess/15min"
SAVE_HOURLY = f"{BASE}/agent-final/outputs/preprocess/hourly"

TARGET = "Usage_kWh"
TEST_RATIO = 0.18 

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder

os.makedirs(SAVE_15MIN, exist_ok=True)
os.makedirs(SAVE_HOURLY, exist_ok=True)

## Part 1 — Load raw data & basic check

In [ ]:
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df.columns = df.columns.str.strip()
df["date"] = pd.to_datetime(df["date"], format="%d/%m/%Y %H:%M")
df = df.sort_values("date").reset_index(drop=True)

print("Shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Missing values:\n", df.isnull().sum())
print("Duplicate dates:", df["date"].duplicated().sum())
df.head()

## Part 2 — Look at distributions

Before modeling we need to see if features are skewed, mostly zero, etc.

In [ ]:
num_cols = [
    "Usage_kWh",
    "Lagging_Current_Reactive.Power_kVarh",
    "Leading_Current_Reactive_Power_kVarh",
    "CO2(tCO2)",
    "Lagging_Current_Power_Factor",
    "Leading_Current_Power_Factor",
    "NSM",
]

print(df[num_cols].describe().round(2))

In [ ]:
# How many zeros? (features that are empty most of the time are useless)
for col in num_cols:
    zero_pct = 100 * (df[col] == 0).mean()
    print(f"{col:45s}  zeros = {zero_pct:.1f}%")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, col in zip(axes.ravel(), num_cols):
    ax.hist(df[col], bins=40, edgecolor="white")
    ax.set_title(col, fontsize=8)
plt.suptitle("Feature distributions (raw)")
plt.tight_layout()
plt.show()

**What we notice:**
- `Leading_Current_Reactive_Power_kVarh` — lots of zeros (~67%)
- `CO2(tCO2)` — mostly zero (~60%)
- `Usage_kWh` — right-skewed (few high consumption spikes)

Zeros alone don't mean drop — we check correlation next.

## Part 3 — Correlation with target (leakage check)

If an **input** feature has correlation ~1 with `Usage_kWh`, the model might cheat (data leakage).

In [ ]:
corr_target = df[num_cols].corr()[TARGET].drop(TARGET).sort_values(ascending=False)
print("Correlation with Usage_kWh:")
print(corr_target.round(4))

In [ ]:
plt.figure(figsize=(8, 4))
corr_target.plot(kind="barh", color="steelblue")
plt.axvline(0.95, color="red", linestyle="--", label="leakage worry line (0.95)")
plt.axvline(-0.95, color="red", linestyle="--")
plt.title("How much each feature correlates with Usage_kWh")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Visual proof — CO2 almost tracks Usage_kWh
sample = df.sample(2000, random_state=42)
plt.figure(figsize=(5, 4))
plt.scatter(sample["CO2(tCO2)"], sample[TARGET], s=8, alpha=0.4)
plt.xlabel("CO2(tCO2)")
plt.ylabel("Usage_kWh")
plt.title(f"CO2 vs Usage_kWh  (r = {corr_target['CO2(tCO2)']:.3f})")
plt.show()

**Our conclusion from Part 3:**
- `CO2(tCO2)` has **r ≈ 0.99** with target → CO2 is *calculated from* energy use, not an independent sensor.
- Using it as input = **target leakage** (model looks accurate but learns nothing useful).
- `Lagging_Current_Reactive.Power_kVarh` has high r (~0.90) but it is a real electrical measurement — we **keep** it.
- We will **drop CO2**.

## Part 4 — Feature vs feature (redundant inputs)

Two inputs that say the same thing confuse the model.

In [ ]:
df["hour"] = df["date"].dt.hour

print("NSM vs hour correlation:", round(df["NSM"].corr(df["hour"]), 4))

sample = df.sample(2000, random_state=42)
plt.figure(figsize=(5, 4))
plt.scatter(sample["NSM"], sample["hour"], s=8, alpha=0.4)
plt.xlabel("NSM")
plt.ylabel("hour of day")
plt.title("NSM is just a time counter — same info as hour")
plt.show()

In [ ]:
r_lr_lpf = df["Leading_Current_Reactive_Power_kVarh"].corr(df["Leading_Current_Power_Factor"])
print("Leading reactive vs Leading power factor:", round(r_lr_lpf, 4))

plt.figure(figsize=(5, 4))
plt.scatter(sample["Leading_Current_Reactive_Power_kVarh"], sample["Leading_Current_Power_Factor"], s=8, alpha=0.3)
plt.xlabel("Leading reactive power")
plt.ylabel("Leading power factor")
plt.title(f"Two leading features overlap (r = {r_lr_lpf:.2f})")
plt.show()

In [ ]:
# Full input correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("All numeric features — who correlates with who?")
plt.tight_layout()
plt.show()

**Our conclusion from Part 4:**
- `NSM` ↔ `hour` has **r ≈ 0.999** → NSM is a 15-minute step counter, not new information. **Drop NSM**, add proper time features later.
- `Leading_Current_Reactive_Power_kVarh` ↔ `Leading_Current_Power_Factor` **r ≈ -0.94** plus 67% zeros → **drop Leading reactive**.
- `WeekStatus` vs `Day_of_week` — we check quickly:

In [ ]:
ws = pd.Categorical(df["WeekStatus"]).codes
dw = pd.Categorical(df["Day_of_week"]).codes
print("WeekStatus vs Day_of_week correlation:", round(np.corrcoef(ws, dw)[0, 1], 4))
print("→ Not redundant. Keep both.")

## Part 5 — Final feature decision (from our analysis above)

| Feature | Why |
|---------|-----|
| DROP `CO2(tCO2)` | r=0.99 with target — leakage |
| DROP `NSM` | r=0.999 with hour — duplicate time info |
| DROP `Leading_Current_Reactive_Power_kVarh` | 67% zeros + redundant with Leading PF |
| KEEP lagging sensors, load type, week/day | useful & not redundant |
| ADD `hour_sin`, `hour_cos` | replace NSM with smooth cyclical time |

In [ ]:
DROP = ["CO2(tCO2)", "Leading_Current_Reactive_Power_kVarh", "NSM"]
CAT_COLS = ["WeekStatus", "Day_of_week", "Load_Type"]

clean = df.drop(columns=DROP + ["hour"])  # hour was only for analysis

# cyclical hour instead of NSM
h = clean["date"].dt.hour + clean["date"].dt.minute / 60
clean["hour_sin"] = np.sin(2 * np.pi * h / 24)
clean["hour_cos"] = np.cos(2 * np.pi * h / 24)

# encode categories to numbers
for col in CAT_COLS:
    clean[col] = OrdinalEncoder().fit_transform(clean[[col]].astype(str))

feature_cols = [c for c in clean.columns if c != "date"]
# target first (needed for LSTM sliding windows)
feature_cols = [TARGET] + [c for c in feature_cols if c != TARGET]

print("Final features:", feature_cols)

In [ ]:
# Check we fixed leakage — no input should have |r| > 0.95 with target now
check = clean[feature_cols].corr()[TARGET].drop(TARGET)
print("Correlations after cleaning:")
print(check.round(4))
print("\nMax |r|:", round(check.abs().max(), 4), "→ OK if below 0.95")

In [ ]:
plt.figure(figsize=(7, 6))
sns.heatmap(clean[feature_cols].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Correlation matrix AFTER dropping bad features")
plt.tight_layout()
plt.show()

## Part 6 — Scale & save (15min + hourly)

- Split by **time** (first 82% train, last 18% test)
- `MinMaxScaler` fit on **train only**, then transform both
- Hourly track = resample to 1 hour (mean for numbers, first for categories)

In [ ]:
def save_preprocessed(data, folder):
    """Simple save: data.csv + scaler.pkl"""
    d = data[["date"] + feature_cols].copy()
    split = int(len(d) * (1 - TEST_RATIO))

    train = d.iloc[:split]
    test = d.iloc[split:]

    scaler = MinMaxScaler()
    scaler.fit(train[feature_cols])

    out = d.copy()
    out[feature_cols] = scaler.transform(d[feature_cols])

    os.makedirs(folder, exist_ok=True)
    out.drop(columns="date").to_csv(os.path.join(folder, "data.csv"), index=False)
    with open(os.path.join(folder, "scaler.pkl"), "wb") as f:
        pickle.dump(scaler, f)

    print(f"Saved {folder}")
    print(f"  rows={len(d)}, train={split}, test={len(d)-split}, features={len(feature_cols)}")

In [ ]:
# 15-minute data (native resolution)
save_preprocessed(clean, SAVE_15MIN)

In [ ]:
# Hourly — resample then save
hourly = clean.set_index("date").resample("h").agg({
    TARGET: "mean",
    "Lagging_Current_Reactive.Power_kVarh": "mean",
    "Lagging_Current_Power_Factor": "mean",
    "Leading_Current_Power_Factor": "mean",
    "hour_sin": "mean",
    "hour_cos": "mean",
    "WeekStatus": "first",
    "Day_of_week": "first",
    "Load_Type": "first",
}).reset_index()

print("15min rows:", len(clean), "→ hourly rows:", len(hourly))
save_preprocessed(hourly, SAVE_HOURLY)

In [ ]:
# quick look at saved 15min data
saved = pd.read_csv(os.path.join(SAVE_15MIN, "data.csv"))
print(saved.head())
print("shape:", saved.shape)

## Done

You now have:
- `outputs/preprocess/15min/data.csv` + `scaler.pkl`
- `outputs/preprocess/hourly/data.csv` + `scaler.pkl`

Use these in the **training notebook** next.